<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 11 · 用 HTTP 管理 Source 与四类制品

你已有一个订单导入应用，希望把证据、知识和工作状态交给独立 PowerContext Server 管理。
本篇只使用 `httpx` 与普通 JSON，依次保存 Source、创建四类制品、读取内容、更新版本和分页。
完成后，可以把相同请求迁移到其他编程语言。

这是一条独立路径，不依赖前十篇的变量、数据或 Python Client。预计 25 分钟，无需模型。
每一步的请求、响应和必要 header 都可展开查看；正文帮助你判断应该观察什么。

**成功标志：** Source 原样读回；四类制品可创建和读取；旧版内容保持不变；条件读取返回 304，
过期更新返回 412，缺少更新条件返回 428；分页得到不同制品。
最后还会验证通用 Memory 制品与日常召回的当前关系。

## 先把 Server 和应用分开运行

在**仓库根目录的终端 A** 执行。这个配置文件使用 SQLite，不启用模型或后台提取；
`POWERCONTEXT_HOME` 将实验数据放进本套教程的独立目录。保持这个终端运行。

```bash
uv sync --locked --group notebooks
POWERCONTEXT_HOME="$PWD/examples/jupyter/.powercontext/http-server" \
  uv run --locked --group notebooks powercontext server run \
  --env-file examples/jupyter/server-http.env.example --host 127.0.0.1 --port 8000
```

在**仓库根目录的终端 B** 启动 Jupyter，然后打开本篇：

```bash
uv run --locked --group notebooks jupyter lab --notebook-dir=examples/jupyter
```

如果 8000 已被占用，为终端 A 改用其他端口，并在启动 Jupyter 前设置
`POWERCONTEXT_CLIENT_SERVER_URL`。也可以连接你自己管理的 Server；远程连接使用 HTTPS。
本篇会创建一个新 Scope，因此启用 Access Control 时，需要有创建 Scope 的权限，以及对新 Scope 的读写权限。
`scope_id` 负责确定数据范围，权限由 Server 另行判断。

连接设置可以放在 `examples/jupyter/.env`，或由启动 Jupyter 的环境提供：

| 变量 | 用途 |
| --- | --- |
| `POWERCONTEXT_CLIENT_SERVER_URL` | 默认 `http://127.0.0.1:8000` |
| `POWERCONTEXT_CLIENT_API_TOKEN` | 服务启用鉴权时的 Bearer token；本地教学配置无需填写 |
| `POWERCONTEXT_NOTEBOOK_ENV_FILE` | 可选，指定另一份配置文件的绝对路径 |

不要把 token 写进单元格。本篇只显示教学请求的正文，不显示 Authorization header。

In [ ]:
import json
import os
from pathlib import Path
from uuid import uuid4

import httpx
from dotenv import load_dotenv

tutorial_directory = Path.cwd() / "examples" / "jupyter"
if not tutorial_directory.is_dir():
    tutorial_directory = Path.cwd()
env_file = Path(os.environ.get("POWERCONTEXT_NOTEBOOK_ENV_FILE", tutorial_directory / ".env")).expanduser()
if "POWERCONTEXT_NOTEBOOK_ENV_FILE" in os.environ and not env_file.is_file():
    raise RuntimeError("请检查 POWERCONTEXT_NOTEBOOK_ENV_FILE 指定的文件是否存在。")
if env_file.is_file():
    load_dotenv(env_file, override=False)

base_url = os.environ.get("POWERCONTEXT_CLIENT_SERVER_URL", "http://127.0.0.1:8000").rstrip("/")
url = httpx.URL(base_url)
if url.scheme not in {"http", "https"} or not url.host or url.userinfo or url.query or url.fragment:
    raise ValueError("Server URL 应为 HTTP(S) 地址；凭据请通过 token 环境变量提供。")
if url.scheme == "http" and url.host not in {"127.0.0.1", "localhost", "::1"}:
    raise ValueError("远程 Server 请使用 HTTPS；本机可使用回环 HTTP 地址。")
token = os.environ.get("POWERCONTEXT_CLIENT_API_TOKEN")
headers = {"Authorization": f"Bearer {token}"} if token else {}
if previous_http := globals().get("http"):
    await previous_http.aclose()
http = httpx.AsyncClient(base_url=base_url, headers=headers, timeout=30)
exchanges = []
print("HTTP 连接已准备。下一步检查独立 Server。")

## 1. 确认服务和所需能力

先检查进程、依赖和启用的制品类型。下面的函数只展示 HTTP 交换，不封装业务操作。
展开“请求与响应 JSON”即可看到正文、ETag、Location 和请求 ID；不会显示 Authorization header。

Notebook 关闭不会停止独立 Server，服务生命周期由启动它的终端管理。

In [ ]:
from html import escape

from IPython.display import HTML
from IPython.display import display as display_html


def inspect_response(response, expected=200):
    method, path = response.request.method, response.request.url.path
    body = response.json() if response.content else None
    record = {
        "request": json.loads(response.request.content) if response.request.content else None,
        "response_headers": {
            key: response.headers[key]
            for key in ("ETag", "Location", "X-PowerContext-Request-ID")
            if key in response.headers
        },
        "response": body,
    }
    label = f"{method} {path} → HTTP {response.status_code} · 请求与响应 JSON"
    display_html(
        HTML(
            f"<details><summary>{escape(label)}</summary><pre>{escape(json.dumps(record, ensure_ascii=False, indent=2))}</pre></details>"
        )
    )
    exchanges.append({"method": method, "path": path, "status": response.status_code})
    assert response.status_code == expected, f"预期 HTTP {expected}，请展开响应检查实际错误。"
    return body


inspect_response(await http.get("/health/live"))
inspect_response(await http.get("/health/ready"))
capabilities = inspect_response(await http.get("/v1/capabilities"))
assert {"memory", "experience", "skill", "handoff"} <= set(capabilities["artifact_families"])
print("Source 与四类制品接口可用。")

## 2. 创建本次实验的 Scope

Server 返回的 Scope ID 是后续所有请求的共同范围。保存本次幂等键，再发一次相同请求，应该回到同一个 Scope。
每次完整运行会生成新的幂等键，避免与其他实验混在一起。

启用权限控制的部署仍会独立检查调用者权限；知道 scope_id 本身不会获得访问权。

In [ ]:
from urllib.parse import quote

scope_payload = {"title": "订单导入应用 · HTTP", "summary": "本篇合成数据", "idempotency_key": f"http-{uuid4().hex}"}
scope = inspect_response(await http.post("/v1/scopes", json=scope_payload), expected=201)
scope_id = scope["scope_id"]
retried = inspect_response(await http.post("/v1/scopes", json=scope_payload), expected=201)
assert retried["scope_id"] == scope_id
scope_path = f"/v1/scopes/{quote(scope_id, safe='')}"
print("后续请求都使用这个由 Server 创建的 Scope。")

## 3. 先保存一份实际观察

旧的金额转换会把 1.999 元截成 199 分。执行这个小例子，再检查“转换前先判断精度”的拒绝条件。
将两项实际结果作为 JSON Source 保存，并通过生成的身份读取它。

`create_source` 只保存原始材料，不会同步生成 Memory 或 Experience。

In [ ]:
from decimal import Decimal

amount = Decimal("1.999")
old_cents = int(amount * 100)
reject_before_conversion = amount * 100 != (amount * 100).to_integral_value()
assert old_cents == 199 and reject_before_conversion
source_content = {"input": "1.999", "old_cents": old_cents, "precision_guard_rejects": reject_before_conversion}
source = inspect_response(
    await http.post(f"{scope_path}/sources", json={"source_type": "content", "content": source_content}), expected=201
)
source_path = f"{scope_path}/sources/content/{quote(source['source_id'], safe='')}"
restored_source = inspect_response(await http.get(source_path))
assert restored_source["content"] == source_content
empty_experiences = inspect_response(await http.get(f"{scope_path}/artifacts/experience"))
assert empty_experiences["items"] == []
print("Source 已保存，当前还没有 Experience。")

## 4. 通过同一个入口创建 Memory 和 Experience

Memory 接收一组 entries，Experience 接收 situation、action、outcome、lesson。
它们使用相同的 POST 路径，并返回同一种制品身份：family、artifact_id、revision。

这里由应用决定提交已经整理的内容，创建后直接成为制品。Server 为直接写入自动记录系统来源，
具体内容则由各 family 校验。Experience 的候选审核流程可在第 06 篇单独体验。

先观察制品本身的创建与读取；第 9 步会进一步验证这份独立 Memory 与日常召回的关系。

In [ ]:
experience_content = {
    "situation": "amount: 三位小数直接转整数分可能被静默截断。",
    "action": "在转换前检查乘以 100 后是否为整数，不符合时明确拒绝。",
    "outcome": "本篇实际输入 1.999 在旧写法中变成 199，新精度条件识别出该输入应被拒绝。",
    "lesson": "amount: 先校验精度再转换为整数分，不能用截断代替输入校验。",
}
contents = {
    "memory": {"entries": [{"kind": "decision", "text": "amount: 金额以整数分保存。"}]},
    "experience": experience_content,
}
created_artifacts = {}
for family, content in contents.items():
    response = await http.post(f"{scope_path}/artifacts", json={"family": family, "content": content})
    created_artifacts[family] = inspect_response(response, expected=201)
    assert created_artifacts[family]["revision"] == 1 and created_artifacts[family]["sources"]
print("Memory 与 Experience 第一版已创建。")

## 5. 继续创建 Skill 和 Handoff

Skill 保存已经整理的操作说明。Handoff 保存“本篇金额精度检查已经完成”这个有限目标的状态，
并明确引用前面的实际 Source。Handoff 是 Scope 单例，已经存在时应通过后续版本更新，而不是再次 Create。

这些调用不会自动执行 Skill，也不会替接收方确认工作现场。相关后续流程分别在第 07、05 篇演示。

In [ ]:
contents = {
    "skill": {
        "name": "check-amount-precision",
        "description": "修改金额转换逻辑时使用。",
        "instructions": "准备正常与超精度输入；检查精度后再转换整数分；执行用例并记录实际结果。",
        "validation": ["正常输入符合预期", "超精度输入被明确拒绝"],
    },
    "handoff": {
        "schema": "powercontext.handoff.v1",
        "objective": "完成本篇 1.999 金额精度行为检查",
        "state": [
            {
                "text": "已观察旧写法截断行为，并验证精度条件能识别该输入。",
                "citations": [
                    {
                        "kind": "source",
                        "source_ref": {"name": "content", "source_id": source["source_id"]},
                    }
                ],
            }
        ],
        "disposition": "complete",
        "next_action": None,
        "omissions": [{"text": "尚未验证其他货币格式、非法文本和数据规模。", "citation": None}],
    },
}
for family, content in contents.items():
    created_artifacts[family] = inspect_response(
        await http.post(f"{scope_path}/artifacts", json={"family": family, "content": content}), expected=201
    )
    assert created_artifacts[family]["revision"] == 1
print("四类制品都已通过统一入口创建。")

## 6. 统一列出和读取，保留真正返回的 ETag

Collection GET 返回摘要，item GET 返回完整内容。使用同一段代码逐类读取，并保存响应 ETag。
ETag 是不透明条件，必须原样传回，不能由 revision 数字自行拼造。

展开 Skill 的响应，可以看到它已经形成标准包；这仍不表示文件已经安装到某个 Host。

In [ ]:
artifact_paths, heads, etags = {}, {}, {}
for family, created in created_artifacts.items():
    page = inspect_response(await http.get(f"{scope_path}/artifacts/{family}", params={"limit": 10}))
    assert any(item["artifact_id"] == created["artifact_id"] for item in page["items"])
    assert all("content" not in item for item in page["items"])
    artifact_paths[family] = f"{scope_path}/artifacts/{family}/{quote(created['artifact_id'], safe='')}"
    response = await http.get(artifact_paths[family])
    heads[family] = inspect_response(response)
    etags[family] = response.headers["ETag"]
    assert heads[family]["revision"] == 1
assert heads["skill"]["content"]["package"] is not None
assert heads["experience"]["content"] == experience_content
print("四类制品的摘要和完整内容均已读取。")

## 7. 更新 Experience，保留历史版本

为经验增加“先覆盖边界输入”的说明，用读取时的 ETag 请求 PUT。
成功后应产生第二版；GET `/revisions/1` 仍读到原始内容。

使用最新 ETag 条件读取，会收到 304，没有响应正文。用旧 ETag 更新会收到 412；
不提供 If-Match 会收到 428。这些都是需要应用明确处理的正常契约行为。

In [ ]:
updated_content = {**experience_content, "lesson": experience_content["lesson"] + " 先用边界输入验证拒绝行为。"}
response = await http.put(
    artifact_paths["experience"], headers={"If-Match": etags["experience"]}, json={"content": updated_content}
)
updated = inspect_response(response)
current_etag = response.headers["ETag"]
assert updated["revision"] == 2 and updated["content"] == updated_content
historical = inspect_response(await http.get(f"{artifact_paths['experience']}/revisions/1"))
assert historical["content"] == experience_content
unchanged = inspect_response(
    await http.get(artifact_paths["experience"], headers={"If-None-Match": current_etag}), expected=304
)
assert unchanged is None
stale = inspect_response(
    await http.put(
        artifact_paths["experience"], headers={"If-Match": etags["experience"]}, json={"content": updated_content}
    ),
    expected=412,
)
assert stale["error"]["code"] == "revision_conflict"
inspect_response(await http.put(artifact_paths["experience"], json={"content": updated_content}), expected=428)
print("第二版已保存，第一版可读，更新条件被 Server 实际检查。")

## 8. 练习：保存另一条经验，再分页读取

刚才实际观察了一次 412。把“发生版本冲突后先重新读取”的处理经验保存成第二个 Experience。
用 limit=1 读取第一页，将返回的 next_cursor 原样传给下一页，检查没有重复或遗漏。

你可以修改第二条经验的措辞再完整运行；不要解析或自行生成 Cursor。

In [ ]:
second = inspect_response(
    await http.post(
        f"{scope_path}/artifacts",
        json={
            "family": "experience",
            "content": {
                "situation": "另一个写入者已经更新了制品。",
                "action": "使用读取时的 ETag 提交，并检查状态码。",
                "outcome": "本篇的旧 ETag 更新被 Server 以 412 拒绝。",
                "lesson": "遇到版本冲突后先重新读取，结合当前内容判断自己的修改意图。",
            },
        },
    ),
    expected=201,
)
first_page = inspect_response(await http.get(f"{scope_path}/artifacts/experience", params={"limit": 1}))
assert len(first_page["items"]) == 1 and first_page["next_cursor"]
second_page = inspect_response(
    await http.get(f"{scope_path}/artifacts/experience", params={"limit": 1, "cursor": first_page["next_cursor"]})
)
assert second_page["next_cursor"] is None
ids = {item["artifact_id"] for page in (first_page, second_page) for item in page["items"]}
assert ids == {created_artifacts["experience"]["artifact_id"], second["artifact_id"]}
print("分页读回两个不同的 Experience。")

## 9. 写入制品之后，哪些知识会被日常召回？

当前实现中，通用 Memory Create 生成独立制品，日常 search/prepare 使用 Scope 的默认 Memory。
所以刚才的 Memory 可以通过制品 API 读取，但仅靠它不能证明日常召回已接通。

先观察搜索为空，再明确把约定写入日常 Memory，重新搜索并准备上下文。
直接创建的 Experience 则可参与 PreparedContext；本例同时检查这两个来源。
这是选择 API 时必须知道的当前边界，不能静默把一种写入当成另一种。

In [ ]:
before = inspect_response(
    await http.post("/v1/memory/search", json={"scope_id": scope_id, "query": "amount", "mode": "fts"})
)
assert before["hits"] == []
daily = inspect_response(
    await http.post(
        "/v1/memory/remember",
        json={
            "scope_id": scope_id,
            "kind": "decision",
            "text": "amount: 金额以整数分保存。",
            "reason": "应用确认此约定需要参与日常召回",
        },
    )
)
daily_id = daily["entry"]["citation"]["memory_ref"]["artifact_id"]
assert daily_id != created_artifacts["memory"]["artifact_id"]
found = inspect_response(
    await http.post("/v1/memory/search", json={"scope_id": scope_id, "query": "amount", "mode": "fts"})
)
assert any(hit["citation"] == daily["entry"]["citation"] for hit in found["hits"])
context = inspect_response(
    await http.post("/v1/context/prepare", json={"scope_id": scope_id, "query": "amount", "max_bytes": 6000})
)
assert context["status"] == "ready"
assert daily_id in context["content"] and created_artifacts["experience"]["artifact_id"] in context["content"]
assert created_artifacts["memory"]["artifact_id"] not in context["content"]
print("日常 Memory 与直接创建的 Experience 都已进入本次上下文。")

## 关闭应用连接

下面关闭本篇 HTTP Client。Server 仍由启动它的终端管理；完成后在终端 A 按 Ctrl+C 停止教学服务。
实验数据保留在独立 SQLite 中。使用已有服务时，本篇创建的 Scope 也会保留在那个服务中。
清理方法见 [README](README.md#清理实验数据)。

In [ ]:
await http.aclose()
print(f"本篇完成 {len(exchanges)} 次 HTTP 交换，Client 已关闭。")

## 将这条路径用于自己的应用

| 已体验的通用操作 | HTTP 路径 |
| --- | --- |
| 创建 Source | `POST /v1/scopes/{scope_id}/sources` |
| 读取 Source | `GET /v1/scopes/{scope_id}/sources/{source_type}/{source_id}` |
| 创建制品 | `POST /v1/scopes/{scope_id}/artifacts` |
| 列出同类制品 | `GET /v1/scopes/{scope_id}/artifacts/{family}` |
| 读取当前版本 | `GET /v1/scopes/{scope_id}/artifacts/{family}/{artifact_id}` |
| 更新版本 | `PUT /v1/scopes/{scope_id}/artifacts/{family}/{artifact_id}` |
| 读取历史版本 | `GET /v1/scopes/{scope_id}/artifacts/{family}/{artifact_id}/revisions/{revision}` |

这七个操作不包含通用 Artifact DELETE 或 Source collection 搜索。
知识的搜索、停用、候选审核、Skill 导出和交接回执分别使用相应业务接口。
完整字段以 [OpenAPI](../../openapi/powercontext.yaml) 为准；当前服务的交互参考位于 `/docs`。

| 现象 | 下一步 |
| --- | --- |
| ConnectError | 检查独立 Server 进程和端口 |
| 401 / 403 | 分别检查身份凭据与操作权限 |
| 404 | 检查 Scope、family、制品 ID 和指定 revision 是否来自当前服务 |
| 412 / 428 | 当前 ETag 已变化或缺少 If-Match，重新读取后判断修改 |
| 422 | 对照 family 对应的内容 schema 修正字段 |
| 503 | 检查 `/health/ready` 和 Server 依赖状态 |

Python Client 的通用方法与这组路径对应。当前返回模型没有暴露 ETag，因此本篇直接从 HTTP 响应头获取它。
选择上层业务接口和 Agent 集成的依据见 [课程设计](DESIGN.md#接口选择)。